In [15]:
import h5py
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

In [16]:
from pycs.sparsity.mrs.mrs_tools import *

In [10]:
def add_shape_noise(kg, sigma_e=0.26, galaxy_density=6.75, nside=512):
    """
    Adds shape noise to a full-sky Healpix convergence (kappa) map.

    Parameters:
    - kg: np.ndarray, the input kappa map
    - sigma_e: float, intrinsic ellipticity dispersion per galaxy
    - galaxy_density: float, galaxy number density per arcmin²
    - nside: int, Healpix resolution parameter

    Returns:
    - noisy_kg: np.ndarray, kappa map with added shape noise
    """
    npix = hp.nside2npix(nside)  # Total number of pixels
    pixel_area_arcmin2 = hp.nside2pixarea(nside, degrees=True) * 3600  # Convert to arcmin²
    # print(f"Pixel area: {pixel_area_arcmin2:.2f} arcmin²")  # Print pixel area for reference
    sigma_pix = sigma_e / np.sqrt(galaxy_density * pixel_area_arcmin2)  # Compute pixel noise
    # print(f"Pixel noise: {sigma_pix:.3f} kappa")  # Print pixel noise for reference

    noise = np.random.normal(loc=0, scale=sigma_pix, size=npix)  # Generate noise
    return kg + noise  # Add noise to kappa map

In [5]:
def get_norm_wtl1_sphere(Map, nscales, nbins=None, Mask=None, min_snr=None, max_snr=None, path="/."):

    # Set default for nbins if not provided
    if nbins is None:
        nbins = 40

    # Perform undecimated wavelet transform on the spherical map
    WT = mrs_uwttrans(Map, verbose=False, path=path)

    # Initialize lists to collect the l1 norm and bins
    l1norm_coll = []
    bins_coll = []

      # Loop through each scale of the wavelet transform
    for i in range(nscales):
        ScaleCoeffs = WT[i]  # Accessing the coefficients for the i-th scale

        # Apply the mask if provided
        if Mask is not None:
            ScaleCoeffs = ScaleCoeffs[Mask != 0]

        # Normalize the wavelet scale to the same energy level
        energy = np.sum(ScaleCoeffs**2)
        normalization_factor = np.sqrt(energy)
        if normalization_factor > 0:
            ScaleCoeffs_normalized = ScaleCoeffs / normalization_factor 
        
        # Set the minimum and maximum values based on inputs or defaults
        min_val = min_snr if min_snr is not None else np.min(ScaleCoeffs_normalized)
        max_val = max_snr if max_snr is not None else np.max(ScaleCoeffs_normalized)

        # Define thresholds and bins
        thresholds = np.linspace(min_val, max_val, nbins + 1)
        bins = 0.5 * (thresholds[:-1] + thresholds[1:])

        # Digitize the values into bins
        digitized = np.digitize(ScaleCoeffs_normalized, thresholds)

        # Calculate the l1 norm for each bin
        bin_l1_norm = [
            np.sum(np.abs(ScaleCoeffs_normalized[digitized == j]))
            for j in range(1, len(thresholds))
        ]

        # Store the bins and l1 norms for this scale
        bins_coll.append(bins)
        l1norm_coll.append(bin_l1_norm)

    # Return the bins and l1 norms for each scale
    return np.array(bins_coll), np.array(l1norm_coll)

In [11]:
def get_norm_wtl1_sphere(Map, nscales, nbins=None, Mask=None, min_snr=None, max_snr=None, path="/."):
    """
    Computes L1 norms of wavelet transform coefficients at different scales.
    """
    # Set default for nbins if not provided
    if nbins is None:
        nbins = 40

    # Perform undecimated wavelet transform on the spherical map
    # Use suppress_stdout to hide the "setting output map dtype" messages

    WT = mrs_uwttrans(Map, nscale=nscales, verbose=False, path=path)

    # Initialize lists to collect the l1 norm and bins
    l1norm_coll = []
    bins_coll = []

    # Loop through each scale of the wavelet transform
    for i in range(nscales):
        ScaleCoeffs = WT[i]  # Accessing the coefficients for the i-th scale

        # Apply the mask if provided
        if Mask is not None:
            ScaleCoeffs = ScaleCoeffs[Mask != 0]

        # Normalize the wavelet scale to the same energy level
        energy = np.sum(ScaleCoeffs**2)
        normalization_factor = np.sqrt(energy)
        if normalization_factor > 0:
            ScaleCoeffs_normalized = ScaleCoeffs / normalization_factor 
        
        # Set the minimum and maximum values based on inputs or defaults
        min_val = min_snr if min_snr is not None else np.min(ScaleCoeffs_normalized)
        max_val = max_snr if max_snr is not None else np.max(ScaleCoeffs_normalized)

        # Define thresholds and bins
        thresholds = np.linspace(min_val, max_val, nbins + 1)
        bins = 0.5 * (thresholds[:-1] + thresholds[1:])

        # Digitize the values into bins
        digitized = np.digitize(ScaleCoeffs_normalized, thresholds)

        # Calculate the l1 norm for each bin
        bin_l1_norm = [
            np.sum(np.abs(ScaleCoeffs_normalized[digitized == j]))
            for j in range(1, len(thresholds))
        ]

        # Store the bins and l1 norms for this scale
        bins_coll.append(bins)
        l1norm_coll.append(bin_l1_norm)

    # Return the bins and l1 norms for each scale
    return np.array(bins_coll), np.array(l1norm_coll)

# Run on all cosmologies

In [3]:
from pycs.sparsity.mrs.mrs_starlet import mrs_uwttrans

In [4]:
import os
import re

In [5]:
base_dir = "/home/tersenov/CosmoGridV1/stage3_forecast/grid/"

filename = "projected_probes_maps_nobaryons512.h5"  # File of interest
perm_dirs = [f"perm_{i:04d}" for i in range(7)]  # "perm_0000" to "perm_0006"

# Find all directories matching "cosmo_XXXXX"
# cosmo_dirs = sorted([d for d in os.listdir(base_dir) if re.match(r"cosmo_\d{6}", d)])
cosmo_dirs = sorted([d for d in os.listdir(base_dir) if d.startswith("cosmo_")])

In [6]:
np.array(cosmo_dirs).shape, perm_dirs

((2502,),
 ['perm_0000',
  'perm_0001',
  'perm_0002',
  'perm_0003',
  'perm_0004',
  'perm_0005',
  'perm_0006'])

In [8]:
import tempfile

# Function to compute L1-norms for a given file
def process_file(file_path):
    """Loads the file, extracts the kappa map, computes L1 norms, and saves results."""
    try:
        with h5py.File(file_path, "r") as f:
            kg = np.array(f["kg/stage3_lensing4"])  # Extract kappa map
        
        # Add shape noise
        kg_noisy = add_shape_noise(kg, sigma_e=0.27, galaxy_density=6.75, nside=512)

        # Create a unique temporary directory per process
        with tempfile.TemporaryDirectory() as tmp_dir:
            # Compute L1 norms using a unique temp path
            bins, l1norms = get_norm_wtl1_sphere(
                kg_noisy, nscales=5, nbins=40, Mask=None, min_snr=-0.003, max_snr=0.003,
                path=tmp_dir  # <-- Ensures unique temp files
            )

            
            
        # Save results
        save_path = file_path.replace(".h5", "_l1_norms_bin4_noisy_s027.npy")  # Change extension
        np.save(save_path, l1norms)
        
        print(f"Processed: {file_path} -> {save_path}")
        return save_path  # For debugging/logging
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


same, but checking if file already exists

In [ ]:
import os
def process_file(file_path):
    """Loads the file, extracts the kappa map, computes L1 norms, and saves results."""
    
    # Define the expected output filename
    save_path = file_path.replace(".h5", "_l1_norms_bin2_noisy_s027.npy")

    # Check if the file already exists; if so, skip processing
    if os.path.exists(save_path):
        print(f"Skipping {file_path}, L1 norm file already exists.")
        return save_path  # Return the path to indicate it was "processed"

    try:
        with h5py.File(file_path, "r") as f:
            kg = np.array(f["kg/stage3_lensing4"])  # Extract kappa map

        # Create a unique temporary directory per process
        with tempfile.TemporaryDirectory() as tmp_dir:
            # Compute L1 norms using a unique temp path
            bins, l1norms = get_norm_wtl1_sphere(
                kg, nscales=5, nbins=40, Mask=None, min_snr=-0.004, max_snr=0.006,
                path=tmp_dir  # <-- Ensures unique temp files
            )
            

        # Save results
        np.save(save_path, l1norms)
        
        print(f"Processed: {file_path} -> {save_path}")
        return save_path
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


In [12]:
process_file('/home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_000001/perm_0000/projected_probes_maps_nobaryons512.h5')

Processed: /home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_000001/perm_0000/projected_probes_maps_nobaryons512.h5 -> /home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_000001/perm_0000/projected_probes_maps_nobaryons512_l1_norms_bin4_noisy_s027.npy


'/home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_000001/perm_0000/projected_probes_maps_nobaryons512_l1_norms_bin4_noisy_s027.npy'

In [10]:
! which python

/home/tersenov/software/cosmostat/cosmo_venv/bin/python


In [12]:
! python --version

Python 3.11.5


In [13]:
# Generate all file paths
file_paths = [
    os.path.join(base_dir, cosmo, perm, filename)
    for cosmo in cosmo_dirs
    for perm in perm_dirs
]
print(len(file_paths))

17514


In [ ]:
import multiprocessing as mp

if __name__ == "__main__":
    with mp.Pool(processes=90) as pool:
        results = pool.map(process_file, file_paths)  # Run in parallel

In [ ]:
# Global temp directory (define once)
TMP_DIR = "/tmp/mrs_temp/"  # Use a fast local disk if possible
os.makedirs(TMP_DIR, exist_ok=True)  # Create it once

def process_file(file_path):
    """Loads the file, extracts the kappa map, computes L1 norms, and saves results."""
    try:
        with h5py.File(file_path, "r") as f:
            kg = np.array(f["kg/stage3_lensing1"])  # Extract kappa map

        # Generate a unique filename inside TMP_DIR
        unique_path = os.path.join(TMP_DIR, f"proc_{os.getpid()}_{os.path.basename(file_path)}")

        # Compute L1 norms using a shared temp directory but unique filenames
        bins, l1norms = get_norm_wtl1_sphere(
            kg, nscales=5, nbins=40, Mask=None, min_snr=-0.004, max_snr=0.006,
            path=unique_path  # Ensure unique files per process
        )

        # Save results
        save_path = file_path.replace(".h5", "_l1_norms.npy")
        np.save(save_path, l1norms)
        
        print(f"Processed: {file_path} -> {save_path}")
        return save_path
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


In [ ]:
process_file('/home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_003031/perm_0000/projected_probes_maps_nobaryons512.h5')

setting the output map dtype to [dtype('<f4')]


Processed: /home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_003031/perm_0000/projected_probes_maps_nobaryons512.h5 -> /home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_003031/perm_0000/projected_probes_maps_nobaryons512_l1_norms.npy


'/home/tersenov/CosmoGridV1/stage3_forecast/grid/cosmo_003031/perm_0000/projected_probes_maps_nobaryons512_l1_norms.npy'

In [ ]:
if __name__ == "__main__":
    with mp.Pool(processes=70) as pool:
        results = pool.map(process_file, file_paths)  # Run in parallel

# Fiducial

In [ ]:
base_dir = "/home/tersenov/CosmoGridV1/stage3_forecast/fiducial/cosmo_fiducial/"

filename = "projected_probes_maps_baryonified512.h5"  # File of interest
perm_dirs = [f"perm_{i:04d}" for i in range(200)]  # "perm_0000" to "perm_0006"

In [ ]:
# Generate all file paths
file_paths = [
    os.path.join(base_dir, perm, filename)
    for perm in perm_dirs
]

In [ ]:
np.array(file_paths).shape

(200,)

In [ ]:
if __name__ == "__main__":
    with mp.Pool(processes=70) as pool:
        results = pool.map(process_file, file_paths)  # Run in parallel